<div class="alert alert-block alert-success">
<b>Notebook for combining and compressing BARRA-p99 files for moving to g/data on v46</b>
</div>

In [1]:
# Imports

import xarray as xr
import glob
import intake
import numpy as np
import seaborn as sns
from scipy import stats
import os
import xesmf as xe
import inspect
import calendar
import pandas as pd
from cat_indices import calc_turbulence_indices, windspeed, VWS, TI1, AbsVort, Ri, TI2, TI3
from xarray.groupers import SeasonResampler

import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import logging
logging.getLogger("flox").setLevel(logging.WARNING)

#from plotting_maps.acs_plotting_maps import plot_acs_hazard_multi, plot_acs_hazard, plot_data, cmap_dict, regions_dict
from matplotlib import colors, cm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

import dask
from dask.distributed import Client
from dask.distributed import progress as dask_progress, wait
from dask.diagnostics import ProgressBar
from tqdm.notebook import tqdm


## Set up:

In [2]:
# Directory paths for files (scratch where they are gdata where they are going)
scratch_path = "/scratch/v46/ls7238/CAT_turbulence"
gdata_path = "/g/data/v46/ls7238/home_data_backup/data/CAT-project"

# Lists of run names to keep to previous file naming conventions
list_evaluation = ['evaluation_BARRA-R_r1i1p1f1',]

list_historical = ['historical_ACCESS-CM2_r4i1p1f1', 
                   'historical_ACCESS-ESM1-5_r6i1p1f1',
                   'historical_CESM2_r11i1p1f1', 
                   'historical_CMCC-ESM2_r1i1p1f1',
                   'historical_EC-Earth3_r1i1p1f1',
                   'historical_MPI-ESM1-2-HR_r1i1p1f1',
                   'historical_NorESM2-MM_r1i1p1f1',]

list_ssp126 = ['ssp126_ACCESS-CM2_r4i1p1f1', 
                 'ssp126_ACCESS-ESM1-5_r6i1p1f1',
                 'ssp126_CESM2_r11i1p1f1',
                 'ssp126_CMCC-ESM2_r1i1p1f1',
                 'ssp126_EC-Earth3_r1i1p1f1',
                 'ssp126_MPI-ESM1-2-HR_r1i1p1f1',
                 'ssp126_NorESM2-MM_r1i1p1f1',]

list_ssp370 = ['ssp370_ACCESS-CM2_r4i1p1f1',
                 'ssp370_ACCESS-ESM1-5_r6i1p1f1',
                 'ssp370_CESM2_r11i1p1f1',
                 'ssp370_CMCC-ESM2_r1i1p1f1',
                 'ssp370_EC-Earth3_r1i1p1f1',
                 'ssp370_MPI-ESM1-2-HR_r1i1p1f1',
                 'ssp370_NorESM2-MM_r1i1p1f1',]

list_ssp585 = ['ssp585_ACCESS-CM2_r4i1p1f1',
                 'ssp585_EC-Earth3_r1i1p1f1']

list_future = list_ssp126 + list_ssp370 + list_ssp585


dict_scenario_runs = {
    "evaluation": list_evaluation,
    "historical": list_historical,
    "ssp126": list_ssp126,
    "ssp370": list_ssp370,
    "ssp585": list_ssp585,}


# Not all variables have 200hPa and 200hPa only has hist and ssp375 - initialise dict of what vars and Plevs need what paths
combinations = [
    # turb index  # P   # Scenario runs
    ("TI1",      250, dict_scenario_runs),
    ("TI2",      250, dict_scenario_runs),
    ("VWS",      250, dict_scenario_runs),
    ("windspeed",250, dict_scenario_runs),
    ("Ri",       250, dict_scenario_runs),
    ("TI1",      200, {"historical": list_historical, "ssp370": list_ssp370}),
    ("windspeed",200, {"historical": list_historical, "ssp370": list_ssp370}),
    ("VWS",      200, {"historical": list_historical, "ssp370": list_ssp370}),
]

## Combine and compress:

### Testing:

In [ ]:
# Test:

turb_idx = "TI1"
plvl = 250
scenario = "historical"
run_list = list_historical

filelist = [
    f"{scratch_path}/{turb_idx}/{plvl}hPa/BARRA-p99/freq-above-p99/"
    f"{turb_idx}-{plvl}hPa-monthly-freq-above-p99_AUS-15_{run}_BOM_BARPA-R_v1-r1_6hr.nc"
    for run in run_list
    if os.path.exists(f"{scratch_path}/{turb_idx}/{plvl}hPa/BARRA-p99/freq-above-p99/"
                      f"{turb_idx}-{plvl}hPa-monthly-freq-above-p99_AUS-15_{run}_BOM_BARPA-R_v1-r1_6hr.nc")
]
print(f"Found {len(filelist)} files:")
for f in filelist: 
    print(f)

outdir = f"{gdata_path}/BARRA-p99/{turb_idx}"
os.makedirs(outdir, exist_ok=True)
outfile = outfile = f"{outdir}/{turb_idx}-{plvl}hPa-{scenario}_combined-monthly-freq-above-BARRA-p99_6hr.nc"

ds = xr.open_mfdataset(filelist, combine="nested", concat_dim="run")
ds = ds.assign_coords({"run": run_list})  # add run names now all combined.
print(ds)
ds.to_netcdf(outfile, encoding={turb_idx: {"zlib": True, "complevel": 6}})

print(f"Saved {outfile}")

# verify
ds_check = xr.open_dataset(outfile)
print(ds_check)


Found 7 files:
/scratch/v46/ls7238/CAT_turbulence/TI1/250hPa/BARRA-p99/freq-above-p99/TI1-250hPa-monthly-freq-above-p99_AUS-15_historical_ACCESS-CM2_r4i1p1f1_BOM_BARPA-R_v1-r1_6hr.nc
/scratch/v46/ls7238/CAT_turbulence/TI1/250hPa/BARRA-p99/freq-above-p99/TI1-250hPa-monthly-freq-above-p99_AUS-15_historical_ACCESS-ESM1-5_r6i1p1f1_BOM_BARPA-R_v1-r1_6hr.nc
/scratch/v46/ls7238/CAT_turbulence/TI1/250hPa/BARRA-p99/freq-above-p99/TI1-250hPa-monthly-freq-above-p99_AUS-15_historical_CESM2_r11i1p1f1_BOM_BARPA-R_v1-r1_6hr.nc
/scratch/v46/ls7238/CAT_turbulence/TI1/250hPa/BARRA-p99/freq-above-p99/TI1-250hPa-monthly-freq-above-p99_AUS-15_historical_CMCC-ESM2_r1i1p1f1_BOM_BARPA-R_v1-r1_6hr.nc
/scratch/v46/ls7238/CAT_turbulence/TI1/250hPa/BARRA-p99/freq-above-p99/TI1-250hPa-monthly-freq-above-p99_AUS-15_historical_EC-Earth3_r1i1p1f1_BOM_BARPA-R_v1-r1_6hr.nc
/scratch/v46/ls7238/CAT_turbulence/TI1/250hPa/BARRA-p99/freq-above-p99/TI1-250hPa-monthly-freq-above-p99_AUS-15_historical_MPI-ESM1-2-HR_r1i1p1f1_BO

In [15]:
test1 = xr.open_dataset("/g/data/v46/ls7238/home_data_backup/data/CAT-project/BARRA-p99/TI1/TI1-250hPa-historical_combined-monthly-freq-above-BARRA-p99_6hr.nc")
test1

<xarray.Dataset> Size: 8GB
Dimensions:  (run: 7, time: 432, lat: 436, lon: 777)
Coordinates:
  * lon      (lon) float64 6kB 88.04 88.19 88.34 88.5 ... 207.6 207.8 207.9
  * lat      (lat) float64 3kB -53.58 -53.42 -53.27 -53.11 ... 13.32 13.48 13.63
  * time     (time) datetime64[ns] 3kB 1979-01-31 1979-02-28 ... 2014-12-31
  * run      (run) <U33 924B 'historical_ACCESS-CM2_r4i1p1f1' ... 'historical...
Data variables:
    TI1      (run, time, lat, lon) float64 8GB ...
Attributes: (12/65)
    axiom_version:             0.1.0
    axiom_schemas_version:     0.1.0
    axiom_schema:              cordex-6H.json
    Conventions:               CF-1.11, ACDD-1.3
    activity_id:               DD
    title:                     Bureau of Meteorology Atmospheric Regional Pro...
    ...                        ...
    geospatial_lon_units:      degrees_east
    history:                   Thu Jun 20 10:54:50 2024: /g/data/access/ngm/m...
    turbulence_index:          TI1
    pressure_level:            250
    Description:               Frequency of TI1 at 250hPa above the 99th perc...
    p99:                       2.531626789279212e-07

In [17]:
print(ds_check.sel(run="historical_ACCESS-ESM1-5_r6i1p1f1"))
print(ds_check.sel(run="historical_MPI-ESM1-2-HR_r1i1p1f1"))


<xarray.Dataset> Size: 1GB
Dimensions:  (time: 432, lat: 436, lon: 777)
Coordinates:
  * lon      (lon) float64 6kB 88.04 88.19 88.34 88.5 ... 207.6 207.8 207.9
  * lat      (lat) float64 3kB -53.58 -53.42 -53.27 -53.11 ... 13.32 13.48 13.63
  * time     (time) datetime64[ns] 3kB 1979-01-31 1979-02-28 ... 2014-12-31
    run      <U33 132B 'historical_ACCESS-ESM1-5_r6i1p1f1'
Data variables:
    TI1      (time, lat, lon) float64 1GB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
Attributes: (12/65)
    axiom_version:             0.1.0
    axiom_schemas_version:     0.1.0
    axiom_schema:              cordex-6H.json
    Conventions:               CF-1.11, ACDD-1.3
    activity_id:               DD
    title:                     Bureau of Meteorology Atmospheric Regional Pro...
    ...                        ...
    geospatial_lon_units:      degrees_east
    history:                   Thu Jun 20 10:54:50 2024: /g/data/access/ngm/m...
    turbulence_index:          TI1
    pressure_level:           

In [18]:
ds_orig = xr.open_dataset(filelist[0])
ds_combined_run = ds_check.sel(run="historical_ACCESS-CM2_r4i1p1f1")
print(ds_orig[turb_idx].mean().values)
print(ds_combined_run[turb_idx].mean().values)


0.002233947950417398
0.002233947950417398


### Compute:

In [3]:
# Cycle through each index and each p-level it has as defined in combination
for turb_idx, plvl, scenario_runs in tqdm(combinations, desc="combination"):
    for scenario, run_list in tqdm(scenario_runs.items(), desc="scenario", leave=False):

        # 200hPa has no BARRA-p99 subfolder
        if plvl == 200:
            freq_scratch_dir = f"{scratch_path}/{turb_idx}/{plvl}hPa/freq-above-p99"
        else:
            freq_scratch_dir = f"{scratch_path}/{turb_idx}/{plvl}hPa/BARRA-p99/freq-above-p99"

        # build filelist and matching run names together so they stay in sync
        filelist, found_runs = [], []
        for run in run_list:
            # scratch file naming convention
            f = f"{freq_scratch_dir}/{turb_idx}-{plvl}hPa-monthly-freq-above-p99_AUS-15_{run}_BOM_BARPA-R_v1-r1_6hr.nc"
            if os.path.exists(f):
                filelist.append(f)
                found_runs.append(run)

        # if no files are found
        if not filelist:
            print(f"No files for {turb_idx} {plvl}hPa {scenario}, skipping")
            continue    # go to next combination

        # If files are found then:
        outdir = f"{gdata_path}/BARRA-p99/{turb_idx}"
        os.makedirs(outdir, exist_ok=True)
        # g/data file naming convention
        outfile = f"{outdir}/{turb_idx}-{plvl}hPa-{scenario}_combined-monthly-freq-above-BARRA-p99_6hr.nc"


        if os.path.exists(outfile):
            #
            print(f"Already exists: {outfile}")
        else:
            ds = xr.open_mfdataset(filelist, combine="nested", concat_dim="run")
            ds = ds.assign_coords({"run": found_runs})
            ds.to_netcdf(outfile, encoding={turb_idx: {"zlib": True, "complevel": 6}})
            print(f"Saved {outfile}")


combination:   0%|          | 0/8 [00:00<?, ?it/s]

scenario:   0%|          | 0/5 [00:00<?, ?it/s]

Already exists: /g/data/v46/ls7238/home_data_backup/data/CAT-project/BARRA-p99/TI1/TI1-250hPa-evaluation_combined-monthly-freq-above-BARRA-p99_6hr.nc
Already exists: /g/data/v46/ls7238/home_data_backup/data/CAT-project/BARRA-p99/TI1/TI1-250hPa-historical_combined-monthly-freq-above-BARRA-p99_6hr.nc
Already exists: /g/data/v46/ls7238/home_data_backup/data/CAT-project/BARRA-p99/TI1/TI1-250hPa-ssp126_combined-monthly-freq-above-BARRA-p99_6hr.nc
Already exists: /g/data/v46/ls7238/home_data_backup/data/CAT-project/BARRA-p99/TI1/TI1-250hPa-ssp370_combined-monthly-freq-above-BARRA-p99_6hr.nc
Already exists: /g/data/v46/ls7238/home_data_backup/data/CAT-project/BARRA-p99/TI1/TI1-250hPa-ssp585_combined-monthly-freq-above-BARRA-p99_6hr.nc


scenario:   0%|          | 0/5 [00:00<?, ?it/s]

Already exists: /g/data/v46/ls7238/home_data_backup/data/CAT-project/BARRA-p99/TI2/TI2-250hPa-evaluation_combined-monthly-freq-above-BARRA-p99_6hr.nc
Already exists: /g/data/v46/ls7238/home_data_backup/data/CAT-project/BARRA-p99/TI2/TI2-250hPa-historical_combined-monthly-freq-above-BARRA-p99_6hr.nc
Already exists: /g/data/v46/ls7238/home_data_backup/data/CAT-project/BARRA-p99/TI2/TI2-250hPa-ssp126_combined-monthly-freq-above-BARRA-p99_6hr.nc
Already exists: /g/data/v46/ls7238/home_data_backup/data/CAT-project/BARRA-p99/TI2/TI2-250hPa-ssp370_combined-monthly-freq-above-BARRA-p99_6hr.nc
Already exists: /g/data/v46/ls7238/home_data_backup/data/CAT-project/BARRA-p99/TI2/TI2-250hPa-ssp585_combined-monthly-freq-above-BARRA-p99_6hr.nc


...

Already exists: /g/data/v46/ls7238/home_data_backup/data/CAT-project/BARRA-p99/TI1/TI1-200hPa-historical_combined-monthly-freq-above-BARRA-p99_6hr.nc
Already exists: /g/data/v46/ls7238/home_data_backup/data/CAT-project/BARRA-p99/TI1/TI1-200hPa-ssp370_combined-monthly-freq-above-BARRA-p99_6hr.nc


scenario:   0%|          | 0/2 [00:00<?, ?it/s]

Already exists: /g/data/v46/ls7238/home_data_backup/data/CAT-project/BARRA-p99/windspeed/windspeed-200hPa-historical_combined-monthly-freq-above-BARRA-p99_6hr.nc
Saved /g/data/v46/ls7238/home_data_backup/data/CAT-project/BARRA-p99/windspeed/windspeed-200hPa-ssp370_combined-monthly-freq-above-BARRA-p99_6hr.nc


scenario:   0%|          | 0/2 [00:00<?, ?it/s]

Saved /g/data/v46/ls7238/home_data_backup/data/CAT-project/BARRA-p99/VWS/VWS-200hPa-historical_combined-monthly-freq-above-BARRA-p99_6hr.nc
Saved /g/data/v46/ls7238/home_data_backup/data/CAT-project/BARRA-p99/VWS/VWS-200hPa-ssp370_combined-monthly-freq-above-BARRA-p99_6hr.nc
